In [32]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [33]:
def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Affiche de manière robuste le nombre de lignes, de variables et l'unité de la cible.
    """
    data = fetch_california_housing()
    X, y = data.data, data.target
    
    nb_lignes, nb_variables = X.shape
    nom_cible = data.target_names[0] if hasattr(data, 'target_names') else "Cible"
    
    # Extraction robuste de la ligne descriptive de la cible
    unite_cible = "Non trouvée textuellement"
    
    for line in data.DESCR.split('\n'):
        # On cherche des mots-clés universels dans la description du dataset
        if "median house value" in line.lower() or nom_cible.lower() in line.lower():
            if ":" in line: # Souvent formaté comme "Nom : description"
                unite_cible = line.split(":", 1)[1].strip()
            else:
                unite_cible = line.strip()
            break

    # Affichage propre
    print(f"California Housing : ({nb_lignes}, {nb_variables}) variables")
    print(f"Cible ({nom_cible}) : {unite_cible}")
    
    return X, y

In [ ]:
X, y = charger_immobilier()
    
# Séparation Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



California Housing : (20640, 8) variables
Cible (MedHouseVal) : The target variable is the median house value for California districts,


In [ ]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.
    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    # Entraînement
    modele.fit(X_train, y_train)
    
    # Prédiction
    y_pred = modele.predict(X_test)
    
    # Calcul des métriques
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    return {"r2": r2, "mae": mae, "rmse": rmse}

In [36]:
# Définition des modèles
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42, n_estimators=100)

In [37]:

# ------------------------------------------
# CAS NORMAL : Dataset complet
# ------------------------------------------
print("\n--- CAS NORMAL ---")
res_lr = evaluer_regression(lr, X_train_scaled, X_test_scaled, y_train, y_test)
res_rf = evaluer_regression(rf, X_train_scaled, X_test_scaled, y_train, y_test)

print(f"LinearRegression : R2={res_lr['r2']:.2f} MAE={res_lr['mae']:.2f} RMSE={res_lr['rmse']:.2f}")
print(f"RandomForest     : R2={res_rf['r2']:.2f} MAE={res_rf['mae']:.2f} RMSE={res_rf['rmse']:.2f}")


--- CAS NORMAL ---
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest     : R2=0.80 MAE=0.33 RMSE=0.51


In [38]:
# ------------------------------------------
# CAS LIMITE : Entraînement sur 100 lignes
# ------------------------------------------
print("\n--- CAS LIMITE (100 lignes) ---")
X_train_lim, y_train_lim = X_train_scaled[:100], y_train[:100]

res_lr_lim = evaluer_regression(lr, X_train_lim, X_test_scaled, y_train_lim, y_test)
res_rf_lim = evaluer_regression(rf, X_train_lim, X_test_scaled, y_train_lim, y_test)

print(f"LinearRegression (Limité) : R2={res_lr_lim['r2']:.2f} MAE={res_lr_lim['mae']:.2f} RMSE={res_lr_lim['rmse']:.2f}")
print(f"RandomForest (Limité)     : R2={res_rf_lim['r2']:.2f} MAE={res_rf_lim['mae']:.2f} RMSE={res_rf_lim['rmse']:.2f}")


--- CAS LIMITE (100 lignes) ---
LinearRegression (Limité) : R2=0.40 MAE=0.54 RMSE=0.88
RandomForest (Limité)     : R2=0.54 MAE=0.56 RMSE=0.77


In [ ]:
# ------------------------------------------
# CAS ADVERSARIAL : Quartier fictif hors-norme
# ------------------------------------------
print("\n--- CAS ADVERSARIAL ---")
valeurs_moyennes = np.mean(X, axis=0)
quartier_fictif = valeurs_moyennes.copy()
quartier_fictif[0] = 0.0    # Revenu médian à 0
quartier_fictif[4] = 9000.0 # Population énorme

quartier_fictif_scaled = scaler.transform([quartier_fictif])

pred_lr = lr.predict(quartier_fictif_scaled)[0]
pred_rf = rf.predict(quartier_fictif_scaled)[0]

print(f"Prédiction LinearRegression : {pred_lr:.2f} (soit {pred_lr*100000:.0f} $)")
print(f"Prédiction RandomForest     : {pred_rf:.2f} (soit {pred_rf*100000:.0f} $)")


--- CAS ADVERSARIAL ---
Prédiction LinearRegression : 0.17 (soit 16913 $)
Prédiction RandomForest     : 1.19 (soit 119458 $)
